# Test of G3 (`03_repository_contributor_concentration`)

This notebook tests `03_repository_contributor_concentration` with small hand-made Silver tables in the schema `github_lakehouse.test_gold_g3`. It runs the real notebook with `%run` and `test_schema = test_gold_g3`, so **production tables are never written**. The last check section proves that.

It has the same sections as the tests of G1, G2, G4 and G6. What is different: the hand-made rows and the expected values are those of G3, so they test the people of a repository-day, the most active ones and the HHI.

**Before you run it:** `08_gold_audit_table` must have run once, and the nine Silver tables must exist. The test copies the definitions of the tables. It does not read their rows.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Four cells are EXPECTED to fail.** The text before each of them says so. "Run all" stops at such a cell. Then click the cell **after** it and choose "Run all below". It does no harm if one of these four cells runs more than once.
- **The result of the whole test is the cell "Result of the whole test" near the end.** It prints `TEST PASSED` only when all 13 sections ran and every check passed. If a section failed or was left out, it stops with `TEST FAILED` and names the section.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: empty Silver tables. T14, T15: a source date without rows in the six tables, although a watch event has that source date. Both give `NO_INPUT` and leave Gold unchanged |
| Schema | T7: the Gold table has exactly the columns and types of the contract |
| Exact values | T7: the cases of the plan, worked out by hand. Three people with 6, 3 and 1 events. Four people with 5, 3, 1 and 1 events. One person with all events. A person with a push, a review and a comment. Four people with the same number of events. An HHI that must be rounded |
| Only the six tables | T7: watch, fork and release events are not counted, are not people, do not give a label, and a repository with only such events has no row. T11: a late watch event does not make its day an affected day |
| Missing values | T7: events without an actor, events without a name, and one event without a date in each of the six tables |
| Zero base | T7: a day whose events all have no actor has the coverage 0, 0 people, and NULL in the top counts, the shares and the HHI. 5 events of one person among 8 attributable and 2 unattributable events give the top share 62.5, not 50 |
| Rerun | T8, T9: a second run changes nothing |
| Late event | T10, T11: new Silver rows change the right Gold rows, and the other days are not touched. The people are ranked for each day on its own |
| Removed event | T12, T13: removed Silver rows lower one Gold row and remove another |
| Failed check | T16 to T18: two failing quality gates leave Gold unchanged, then a recovery run |
| A day that is gone | T19, T20: a run for all dates removes a day that no longer exists in Silver |
| Bad parameter | T21 to T23: an invalid `source_date`, `test_schema` or `run_id` stops the notebook in its parameter step. T24: nothing was written |
| Production safety | T24: no production table changed |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean.

`CREATE TABLE ... LIKE` makes an empty table with the same columns as another table. The Silver test tables are empty copies of the real Silver tables, and the audit test table is an empty copy of the real audit table. The test tables have the same names as the real ones, only in the test schema.

The notebook reads six Silver tables. The test also creates the watch, fork and release tables and puts rows into them, to prove that the notebook does not count them.

The Gold test table is created by the notebook itself, in its first run.

In [0]:
TEST_SCHEMA = "test_gold_g3"

TEST_SILVER = f"github_lakehouse.{TEST_SCHEMA}"
PROD_SILVER = "github_lakehouse.silver"

# The six Silver tables that the notebook reads, and the three it does not read.
SILVER_TABLE_NAMES = [
    "push_events",
    "pull_request_events",
    "issue_events",
    "issue_comment_events",
    "pull_request_review_events",
    "pull_request_review_comment_events",
    "watch_events",
    "fork_events",
    "release_events",
]

# The Silver test tables.
TEST_PUSH = TEST_SILVER + ".push_events"
TEST_PULL_REQUEST = TEST_SILVER + ".pull_request_events"
TEST_ISSUE = TEST_SILVER + ".issue_events"
TEST_ISSUE_COMMENT = TEST_SILVER + ".issue_comment_events"
TEST_REVIEW = TEST_SILVER + ".pull_request_review_events"
TEST_REVIEW_COMMENT = TEST_SILVER + ".pull_request_review_comment_events"
TEST_WATCH = TEST_SILVER + ".watch_events"
TEST_FORK = TEST_SILVER + ".fork_events"
TEST_RELEASE = TEST_SILVER + ".release_events"

TEST_GOLD = f"github_lakehouse.{TEST_SCHEMA}.repository_contributor_concentration"
TEST_STAGING = f"github_lakehouse.{TEST_SCHEMA}.repository_contributor_concentration_staging"
TEST_AUDIT = f"github_lakehouse.{TEST_SCHEMA}.gold_pipeline_runs"
TEST_COPY = f"github_lakehouse.{TEST_SCHEMA}.gold_rows_after_run_1"

PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_GOLD = "github_lakehouse.gold.repository_contributor_concentration"
PROD_STAGING = "github_lakehouse.gold_staging.repository_contributor_concentration"

# Every test row fills the same seven columns. Every Silver table has them.
# The other columns of the Silver tables stay NULL.
SEVEN_COLUMNS = "(event_id, event_timestamp, event_date, actor_id, repo_id, repo_name, _source_date)"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Every check section writes its number of failed checks into this dictionary.
# The cell "Result of the whole test" reads it at the end.
section_results = {}

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

for table_name in SILVER_TABLE_NAMES:
    if not spark.catalog.tableExists(PROD_SILVER + "." + table_name):
        raise RuntimeError(
            f"{PROD_SILVER}.{table_name} does not exist. Run the Silver notebook of this table first."
        )

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

for table_name in SILVER_TABLE_NAMES:
    spark.sql(f"CREATE TABLE {TEST_SILVER}.{table_name} LIKE {PROD_SILVER}.{table_name}")

spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The production Gold table and its staging copy may not exist yet. The value `-1` means: the table does not exist.

In [0]:
prod_audit_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_before = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_before = -1

print("Production audit version  :", prod_audit_version_before)
print("Production Gold version   :", prod_gold_version_before)
print("Production staging version:", prod_staging_version_before)

### A two-line summary of every audit row

An audit row holds two maps: the Silver rows read per table, and the quality numbers. The checks below compare them as text. This view writes them into two readable lines, for example:

`push=20 pull_requests=6 issues=2 issue_comments=8 reviews=5 review_comments=1`

`repository_days=7 all=42 attributable=38 unattributable=4 actors=13 top=25 top_3=37 without_attributable=1 single_actor=2`

The view is not a copy. It always reads the audit table as it is at that moment.

In [0]:
audit_summary = spark.sql(f"""
    SELECT
        run_id,
        concat(
            'push=',             CAST(source_rows_read['push_events'] AS STRING),
            ' pull_requests=',   CAST(source_rows_read['pull_request_events'] AS STRING),
            ' issues=',          CAST(source_rows_read['issue_events'] AS STRING),
            ' issue_comments=',  CAST(source_rows_read['issue_comment_events'] AS STRING),
            ' reviews=',         CAST(source_rows_read['pull_request_review_events'] AS STRING),
            ' review_comments=', CAST(source_rows_read['pull_request_review_comment_events'] AS STRING)
        ) AS source_summary,
        concat(
            'repository_days=',        CAST(quality_metrics['repository_days'] AS STRING),
            ' all=',                   CAST(quality_metrics['engineering_activity_events'] AS STRING),
            ' attributable=',          CAST(quality_metrics['attributable_events'] AS STRING),
            ' unattributable=',        CAST(quality_metrics['unattributable_events'] AS STRING),
            ' actors=',                CAST(quality_metrics['unique_active_actors'] AS STRING),
            ' top=',                   CAST(quality_metrics['top_actor_events'] AS STRING),
            ' top_3=',                 CAST(quality_metrics['top_3_actor_events'] AS STRING),
            ' without_attributable=',  CAST(quality_metrics['rows_without_attributable_events'] AS STRING),
            ' single_actor=',          CAST(quality_metrics['single_actor_rows'] AS STRING)
        ) AS metrics_summary
    FROM {TEST_AUDIT}
""")
audit_summary.createOrReplaceTempView("audit_summary")

## T3. Run on the empty Silver tables

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="" $run_id="test-dry-run"

## T4. Checks: no input

Empty Silver tables have no days to rebuild. The run must end as `NO_INPUT`, publish nothing and still write its audit row. The run has created the Gold test table, without rows.

The first six checks compare the table names the notebook used with the names of the test tables: one check per Silver table.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test push table' AS check_name,
               '{PUSH_TABLE}' AS actual, '{TEST_PUSH}' AS expected
        UNION ALL
        SELECT '02 the notebook read the test pull request table', '{PULL_REQUEST_TABLE}', '{TEST_PULL_REQUEST}'
        UNION ALL
        SELECT '03 the notebook read the test issue table', '{ISSUE_TABLE}', '{TEST_ISSUE}'
        UNION ALL
        SELECT '04 the notebook read the test issue comment table', '{ISSUE_COMMENT_TABLE}', '{TEST_ISSUE_COMMENT}'
        UNION ALL
        SELECT '05 the notebook read the test review table', '{REVIEW_TABLE}', '{TEST_REVIEW}'
        UNION ALL
        SELECT '06 the notebook read the test review comment table', '{REVIEW_COMMENT_TABLE}', '{TEST_REVIEW_COMMENT}'
        UNION ALL
        SELECT '07 the notebook wrote the test Gold table', '{GOLD_TABLE}', '{TEST_GOLD}'
        UNION ALL
        SELECT '08 the notebook wrote the test staging table', '{STAGING_TABLE}', '{TEST_STAGING}'
        UNION ALL
        SELECT '09 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '10 Gold rows', CAST(COUNT(*) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 audit rows', CAST(COUNT(*) AS STRING), '1' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '12 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '13 audit pipeline_name', MAX(pipeline_name), 'gold_repository_contributor_concentration'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '14 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '15 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '16 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '17 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '18 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '19 audit Silver rows read', MAX(source_summary), 'push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 review_comments=0'
        FROM audit_summary WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '20 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T4"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made Silver rows

The rows are loaded in three cells:

| Cell | Rows | What it is |
|---|---|---|
| T5.1 | 34 | Repositories 100, 110 and 120 on 1 February: the cases of the plan. Event IDs ...01 to ...34 |
| T5.2 | 44 | Small cases, written row by row. Event IDs ...35 to ...77 and ...79 |
| T5.3 | 25,001 | Repository 700 on 5 February: a very busy repository |

Every event ID starts with `330000000` and exists only once in the nine tables together. Only the seven columns that matter here are filled, the others stay NULL.

### T5.1 Repositories 100, 110 and 120 on 1 February: the cases of the plan

**Repository 100: three people with 6, 3 and 1 events.**

| Person | Events | Kinds | Event IDs |
|---|---|---|---|
| 1 | 6 | 3 pushes, 2 pull requests, 1 review | ...01 to ...06 |
| 2 | 3 | 2 issue comments, 1 inline review comment | ...07 to ...09 |
| 3 | 1 | 1 issue at 22:00, with the new name `org/alpha-renamed` | ...10 |

Repository 100 also has a release at 23:00 (...11, person 1, name `org/alpha-release`), two watch events (...12, ...13, persons 4 and 5) and a fork (...14, person 6). They are not engineering work: they must not be counted, persons 4, 5 and 6 are not people of this table, and the release is not the latest event for the label.

**Repository 110: four people with 5, 3, 1 and 1 events.** Person 9 pushes 5 times (...15 to ...19), person 2 opens 3 pull requests (...20 to ...22), person 8 opens an issue (...23) and person 1 comments (...24). Persons 1 and 2 are also active in repository 100. People are counted per repository, so they count in both. The most active person has the largest ID here, so a ranking by ID instead of by events would give another top 3.

**Repository 120: two people with 5 and 3 events, plus 2 events without an actor.** Person 10 pushes 5 times (...25 to ...29), person 11 reviews 3 times (...30 to ...32), and two issue comments have no actor (...33, ...34).

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 100: person 1
    ('33000000001', TIMESTAMP'2024-02-01 01:00:00', DATE'2024-02-01', 1,    100, 'org/alpha',         DATE'2024-02-01'),
    ('33000000002', TIMESTAMP'2024-02-01 01:00:00', DATE'2024-02-01', 1,    100, 'org/alpha',         DATE'2024-02-01'),
    ('33000000003', TIMESTAMP'2024-02-01 01:00:00', DATE'2024-02-01', 1,    100, 'org/alpha',         DATE'2024-02-01'),
    -- Repository 110: person 9
    ('33000000015', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 9,    110, 'org/beta',          DATE'2024-02-01'),
    ('33000000016', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 9,    110, 'org/beta',          DATE'2024-02-01'),
    ('33000000017', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 9,    110, 'org/beta',          DATE'2024-02-01'),
    ('33000000018', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 9,    110, 'org/beta',          DATE'2024-02-01'),
    ('33000000019', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 9,    110, 'org/beta',          DATE'2024-02-01'),
    -- Repository 120: person 10
    ('33000000025', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 10,   120, 'org/gamma',         DATE'2024-02-01'),
    ('33000000026', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 10,   120, 'org/gamma',         DATE'2024-02-01'),
    ('33000000027', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 10,   120, 'org/gamma',         DATE'2024-02-01'),
    ('33000000028', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 10,   120, 'org/gamma',         DATE'2024-02-01'),
    ('33000000029', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 10,   120, 'org/gamma',         DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_PULL_REQUEST + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 100: person 1
    ('33000000004', TIMESTAMP'2024-02-01 02:00:00', DATE'2024-02-01', 1,    100, 'org/alpha',         DATE'2024-02-01'),
    ('33000000005', TIMESTAMP'2024-02-01 02:00:00', DATE'2024-02-01', 1,    100, 'org/alpha',         DATE'2024-02-01'),
    -- Repository 110: person 2
    ('33000000020', TIMESTAMP'2024-02-01 10:00:00', DATE'2024-02-01', 2,    110, 'org/beta',          DATE'2024-02-01'),
    ('33000000021', TIMESTAMP'2024-02-01 10:00:00', DATE'2024-02-01', 2,    110, 'org/beta',          DATE'2024-02-01'),
    ('33000000022', TIMESTAMP'2024-02-01 10:00:00', DATE'2024-02-01', 2,    110, 'org/beta',          DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_ISSUE + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 100: person 3, the latest event of the six tables, with a new name
    ('33000000010', TIMESTAMP'2024-02-01 22:00:00', DATE'2024-02-01', 3,    100, 'org/alpha-renamed', DATE'2024-02-01'),
    -- Repository 110: person 8
    ('33000000023', TIMESTAMP'2024-02-01 11:00:00', DATE'2024-02-01', 8,    110, 'org/beta',          DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 100: person 2
    ('33000000007', TIMESTAMP'2024-02-01 04:00:00', DATE'2024-02-01', 2,    100, 'org/alpha',         DATE'2024-02-01'),
    ('33000000008', TIMESTAMP'2024-02-01 04:00:00', DATE'2024-02-01', 2,    100, 'org/alpha',         DATE'2024-02-01'),
    -- Repository 110: person 1
    ('33000000024', TIMESTAMP'2024-02-01 12:00:00', DATE'2024-02-01', 1,    110, 'org/beta',          DATE'2024-02-01'),
    -- Repository 120: two comments without an actor
    ('33000000033', TIMESTAMP'2024-02-01 15:00:00', DATE'2024-02-01', NULL, 120, 'org/gamma',         DATE'2024-02-01'),
    ('33000000034', TIMESTAMP'2024-02-01 15:00:00', DATE'2024-02-01', NULL, 120, 'org/gamma',         DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_REVIEW + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 100: person 1
    ('33000000006', TIMESTAMP'2024-02-01 03:00:00', DATE'2024-02-01', 1,    100, 'org/alpha',         DATE'2024-02-01'),
    -- Repository 120: person 11
    ('33000000030', TIMESTAMP'2024-02-01 14:00:00', DATE'2024-02-01', 11,   120, 'org/gamma',         DATE'2024-02-01'),
    ('33000000031', TIMESTAMP'2024-02-01 14:00:00', DATE'2024-02-01', 11,   120, 'org/gamma',         DATE'2024-02-01'),
    ('33000000032', TIMESTAMP'2024-02-01 14:00:00', DATE'2024-02-01', 11,   120, 'org/gamma',         DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_REVIEW_COMMENT + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 100: person 2
    ('33000000009', TIMESTAMP'2024-02-01 05:00:00', DATE'2024-02-01', 2,    100, 'org/alpha',         DATE'2024-02-01')
""")

# Not engineering work: these rows must not be counted.
spark.sql("INSERT INTO " + TEST_RELEASE + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000011', TIMESTAMP'2024-02-01 23:00:00', DATE'2024-02-01', 1,    100, 'org/alpha-release', DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000012', TIMESTAMP'2024-02-01 07:00:00', DATE'2024-02-01', 4,    100, 'org/alpha',         DATE'2024-02-01'),
    ('33000000013', TIMESTAMP'2024-02-01 07:30:00', DATE'2024-02-01', 5,    100, 'org/alpha',         DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000014', TIMESTAMP'2024-02-01 08:00:00', DATE'2024-02-01', 6,    100, 'org/alpha',         DATE'2024-02-01')
""")

print("Rows of repositories 100, 110 and 120 loaded.")

### T5.2 The small cases

| Repository | Day | Events | What it tests |
|---|---|---|---|
| 200 | 1 Feb | 4 | One person (7) with 4 pushes (...35 to ...38): top share 100, HHI 1 |
| 200 | none | 7 | One event without event time and date in **each** of the six tables (...39 to ...44), and a watch event without a date (...45). They cannot be placed on a day. Only the six are counted as left out |
| 200 | 2 Feb | 3 | The same repository on a second day, with a new name. Person 7 pushes, reviews and comments (...46 to ...48): 3 events, 1 person |
| 300 | 2 Feb | 2 | A push and a comment without actor and without name (...49, ...50): coverage 0, 0 people, NULL top counts, shares and HHI, no label |
| 400 | 2 Feb | 3 | Only a watch, a fork and a release (...51 to ...53): no Gold row at all |
| 400 | 3 Feb | 1 | A watch event with the source date 3 February (...54). It is not engineering work, so a run for that source date has no input |
| 500 | 5 Feb | 16 | Four people with 4 events each (...55 to ...70): the ranking has ties at every place, and the numbers do not depend on who is ranked first |
| 510 | 5 Feb | 3 | Three people with 1 event each (...71 to ...73): an HHI of 1 ÷ 3 that must be rounded |
| 600 | 5 Feb | 4 | The label. A pull request (...74, `org/eta-old`) and a push (...75, `org/eta-new`) share the same second. A later inline review comment (...76) has no name and no actor. The latest event of the day is a watch event (...77, `org/eta-watch`), which is not read. The coverage is 2 of 3, which must be rounded |
| 710 | 5 Feb | 1 | A single push (...79). Many real rows look like this |

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 200 on 1 February: one person, 4 pushes
    ('33000000035', TIMESTAMP'2024-02-01 16:00:00', DATE'2024-02-01', 7,    200, 'org/delta',         DATE'2024-02-01'),
    ('33000000036', TIMESTAMP'2024-02-01 16:00:00', DATE'2024-02-01', 7,    200, 'org/delta',         DATE'2024-02-01'),
    ('33000000037', TIMESTAMP'2024-02-01 16:00:00', DATE'2024-02-01', 7,    200, 'org/delta',         DATE'2024-02-01'),
    ('33000000038', TIMESTAMP'2024-02-01 16:00:00', DATE'2024-02-01', 7,    200, 'org/delta',         DATE'2024-02-01'),
    -- Repository 200: an event without event time and event date
    ('33000000039', NULL,                           NULL,             7,    200, 'org/delta',         DATE'2024-02-01'),
    -- Repository 200 on 2 February: a new name
    ('33000000046', TIMESTAMP'2024-02-02 09:00:00', DATE'2024-02-02', 7,    200, 'org/delta-renamed', DATE'2024-02-02'),
    -- Repository 300 on 2 February: no actor, no name
    ('33000000049', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', NULL, 300, NULL,                DATE'2024-02-02'),
    -- Repository 500 on 5 February: person 20
    ('33000000055', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 20,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000056', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 20,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000057', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 20,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000058', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 20,   500, 'org/epsilon',       DATE'2024-02-05'),
    -- Repository 510 on 5 February: person 30
    ('33000000071', TIMESTAMP'2024-02-05 05:00:00', DATE'2024-02-05', 30,   510, 'org/zeta',          DATE'2024-02-05'),
    -- Repository 600 on 5 February: same second as ...74, but the larger event_id
    ('33000000075', TIMESTAMP'2024-02-05 10:00:00', DATE'2024-02-05', 11,   600, 'org/eta-new',       DATE'2024-02-05'),
    -- Repository 710 on 5 February: a single push
    ('33000000079', TIMESTAMP'2024-02-05 15:00:00', DATE'2024-02-05', 50,   710, 'org/eta-single',    DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_PULL_REQUEST + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000040', NULL,                           NULL,             7,    200, 'org/delta',         DATE'2024-02-01'),
    -- Repository 500 on 5 February: person 21
    ('33000000059', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 21,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000060', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 21,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000061', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 21,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000062', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 21,   500, 'org/epsilon',       DATE'2024-02-05'),
    -- Repository 600 on 5 February: same second as ...75, but the smaller event_id
    ('33000000074', TIMESTAMP'2024-02-05 10:00:00', DATE'2024-02-05', 10,   600, 'org/eta-old',       DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_ISSUE + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000041', NULL,                           NULL,             7,    200, 'org/delta',         DATE'2024-02-01'),
    -- Repository 500 on 5 February: person 22
    ('33000000063', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', 22,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000064', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', 22,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000065', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', 22,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000066', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', 22,   500, 'org/epsilon',       DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000042', NULL,                           NULL,             7,    200, 'org/delta',         DATE'2024-02-01'),
    ('33000000048', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', 7,    200, 'org/delta-renamed', DATE'2024-02-02'),
    ('33000000050', TIMESTAMP'2024-02-02 10:30:00', DATE'2024-02-02', NULL, 300, NULL,                DATE'2024-02-02'),
    -- Repository 510 on 5 February: person 31
    ('33000000072', TIMESTAMP'2024-02-05 05:10:00', DATE'2024-02-05', 31,   510, 'org/zeta',          DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_REVIEW + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000043', NULL,                           NULL,             7,    200, 'org/delta',         DATE'2024-02-01'),
    ('33000000047', TIMESTAMP'2024-02-02 09:30:00', DATE'2024-02-02', 7,    200, 'org/delta-renamed', DATE'2024-02-02'),
    -- Repository 510 on 5 February: person 32
    ('33000000073', TIMESTAMP'2024-02-05 05:20:00', DATE'2024-02-05', 32,   510, 'org/zeta',          DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_REVIEW_COMMENT + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000044', NULL,                           NULL,             7,    200, 'org/delta',         DATE'2024-02-01'),
    -- Repository 500 on 5 February: person 23
    ('33000000067', TIMESTAMP'2024-02-05 04:00:00', DATE'2024-02-05', 23,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000068', TIMESTAMP'2024-02-05 04:00:00', DATE'2024-02-05', 23,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000069', TIMESTAMP'2024-02-05 04:00:00', DATE'2024-02-05', 23,   500, 'org/epsilon',       DATE'2024-02-05'),
    ('33000000070', TIMESTAMP'2024-02-05 04:00:00', DATE'2024-02-05', 23,   500, 'org/epsilon',       DATE'2024-02-05'),
    -- Repository 600 on 5 February: a later event without a name and without an actor
    ('33000000076', TIMESTAMP'2024-02-05 12:00:00', DATE'2024-02-05', NULL, 600, NULL,                DATE'2024-02-05')
""")

# Not engineering work: these rows must not be counted.
spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000045', NULL,                           NULL,             7,    200, 'org/delta',         DATE'2024-02-01'),
    ('33000000051', TIMESTAMP'2024-02-02 11:00:00', DATE'2024-02-02', 12,   400, 'org/stars',         DATE'2024-02-02'),
    ('33000000054', TIMESTAMP'2024-02-03 09:00:00', DATE'2024-02-03', 12,   400, 'org/stars',         DATE'2024-02-03'),
    -- Repository 600 on 5 February: the latest event of the day, but not engineering work
    ('33000000077', TIMESTAMP'2024-02-05 13:00:00', DATE'2024-02-05', 15,   600, 'org/eta-watch',     DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000052', TIMESTAMP'2024-02-02 11:10:00', DATE'2024-02-02', 13,   400, 'org/stars',         DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_RELEASE + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000053', TIMESTAMP'2024-02-02 11:20:00', DATE'2024-02-02', 14,   400, 'org/stars',         DATE'2024-02-02')
""")

print("Small cases loaded.")

### T5.3 Repository 700 on 5 February: 25,000 pushes

One account pushes 25,000 times in one day, and one other person writes a review (...78). Such rows exist in the real week: the busiest repository has 64,783 events on one day.

It tests the rounding at the top end:

- The top share is 100 × 25,000 ÷ 25,001 = 99.996. Rounded to 2 decimals that is 100.00, although a second person is active.
- The HHI is (25,000² + 1²) ÷ 25,001² = 625,000,001 ÷ 625,050,001 = 0.99992. Rounded to 4 decimals that is 0.9999.

Both are correct, and no quality gate may fail for them. The count columns hold the exact values.

Nobody wants to write 25,000 lines. `range(25000)` is a small table with 25,000 rows: the numbers 0 to 24,999 in the column `id`. So `SELECT ... FROM range(25000)` gives 25,000 rows, and `33000100000 + id` gives each of them its own event ID: 33000100000 to 33000124999.

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
    SELECT CAST(33000100000 + id AS STRING), TIMESTAMP'2024-02-05 08:00:00', DATE'2024-02-05', 40, 700, 'org/busy', DATE'2024-02-05'
    FROM range(25000)
""")

spark.sql("INSERT INTO " + TEST_REVIEW + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000078', TIMESTAMP'2024-02-05 09:00:00', DATE'2024-02-05', 41,   700, 'org/busy',          DATE'2024-02-05')
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

### A two-line summary of every Gold row

The checks below compare whole Gold rows. This view writes the value columns of a row into two readable lines, for example:

`all=10 attributable=8 unattributable=2 coverage=80.0 actors=2`

`top=5 top_share=62.5 top_3=8 top_3_share=100.0 hhi=0.5313 name=org/gamma`

The first line holds the events and the people, the second line the most active people, the HHI and the name. A NULL value is written as `NULL`.

The view is not a copy. It always reads the Gold table as it is at that moment.

In [0]:
gold_summary = spark.sql(f"""
    SELECT
        repo_id,
        activity_date,
        concat(
            'all=',              CAST(engineering_activity_events AS STRING),
            ' attributable=',    CAST(attributable_events AS STRING),
            ' unattributable=',  CAST(unattributable_events AS STRING),
            ' coverage=',        COALESCE(CAST(actor_coverage_pct AS STRING), 'NULL'),
            ' actors=',          CAST(unique_active_actors AS STRING)
        ) AS count_summary,
        concat(
            'top=',              COALESCE(CAST(top_actor_events AS STRING), 'NULL'),
            ' top_share=',       COALESCE(CAST(top_actor_share_pct AS STRING), 'NULL'),
            ' top_3=',           COALESCE(CAST(top_3_actor_events AS STRING), 'NULL'),
            ' top_3_share=',     COALESCE(CAST(top_3_actor_share_pct AS STRING), 'NULL'),
            ' hhi=',             COALESCE(CAST(hhi_actor_activity AS STRING), 'NULL'),
            ' name=',            COALESCE(repo_name, 'NULL')
        ) AS top_summary
    FROM {TEST_GOLD}
""")
gold_summary.createOrReplaceTempView("gold_summary")

## T6. Run 1: all dates

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="" $run_id="test-run1"

## T7. Checks of run 1: schema and exact values

Worked out by hand.

**The run.** The six tables have 25,069 rows. 25,063 of them have an event date, on 3 days. The six rows without a date (...39 to ...44) are left out and counted. The watch, fork and release rows are not read at all, so the watch event without a date is not counted either. The 25,063 rows belong to 11 repository-days.

**Repository 100 on 1 February: three people with 6, 3 and 1 events.**

- 10 events, all with an actor. Coverage 100.
- Persons 4, 5 and 6 only starred or forked the repository. They are not people of this table, so the repository has 3 people.
- Top actor: 6 events. Share: 100 × 6 ÷ 10 = 60.
- Top 3: 6 + 3 + 1 = 10, everybody. Share 100.
- HHI: (6² + 3² + 1²) ÷ 10² = (36 + 9 + 1) ÷ 100 = 0.46.
- Label: the latest event of the six tables is the issue at 22:00, so the label is `org/alpha-renamed`. The release at 23:00 is later, but this table does not read it.

**Repository 110 on 1 February: four people with 5, 3, 1 and 1 events.**

- Top actor: 5 of 10, share 50.
- Top 3: 5 + 3 + 1 = 9, share 90. The fourth person is left out. Both people with 1 event are tied, and either of them gives 9.
- HHI: (25 + 9 + 1 + 1) ÷ 100 = 0.36.

**Repository 120 on 1 February: the base of the shares.**

- 10 events. 8 have an actor, 2 have none. Coverage: 100 × 8 ÷ 10 = 80.
- 2 people with 5 and 3 events.
- Top share: 100 × 5 ÷ **8** = 62.5. The base is the attributable events. With all 10 events as the base it would be 50, which is wrong.
- Top 3: 5 + 3 = 8, share 100.
- HHI: (25 + 9) ÷ 8² = 34 ÷ 64 = 0.53125, rounded to 4 decimals 0.5313.

**The other rows.**

| Row | Events | People | Top | Top 3 | HHI | Label |
|---|---|---|---|---|---|---|
| 200 on 1 Feb | 4, all by person 7 | 1 | 4, share 100 | 4, share 100 | 1 | `org/delta` |
| 200 on 2 Feb | a push, a review and a comment, all by person 7 | 1 | 3, share 100 | 3, share 100 | 1 | `org/delta-renamed` |
| 300 on 2 Feb | 2, none with an actor. Coverage 0 | 0 | NULL | NULL | NULL | NULL |
| 500 on 5 Feb | 16: four people with 4 each | 4 | 4, share 25 | 12, share 75 | 4 × (4 ÷ 16)² = 0.25 | `org/epsilon` |
| 510 on 5 Feb | 3: three people with 1 each | 3 | 1, share 33.33 | 3, share 100 | 3 × (1 ÷ 3)² = 0.3333 | `org/zeta` |
| 600 on 5 Feb | 3: persons 10 and 11 have 1 each, 1 has no actor. Coverage 100 × 2 ÷ 3 = 66.67 | 2 | 1, share 50 | 2, share 100 | (1 + 1) ÷ 4 = 0.5 | `org/eta-new` |
| 700 on 5 Feb | 25,001 | 2 | 25,000, share 100 | 25,001, share 100 | 0.9999 | `org/busy` |
| 710 on 5 Feb | 1 | 1 | 1, share 100 | 1, share 100 | 1 | `org/eta-single` |

- Repository 300: there is nobody to rank, so the top counts, the shares and the HHI are NULL, not 0. The coverage is 0, because the day has events.
- Repository 400 has only a watch, a fork and a release, so it has no row.
- Repository 500: the four people are tied. Whichever of them is ranked first, the numbers are the same.
- Repository 600, the label: the inline review comment at 12:00 has no name, so it is skipped. The pull request and the push share the same second. The larger `event_id` is ...75, the push, so the label is `org/eta-new`. The watch at 13:00 is later, but not read.
- Repository 700: explained in T5.3.

**The totals of the run.**

| Table | Rows with a date | Where they come from |
|---|---|---|
| Push | 25,026 | 3 + 5 + 5 + 4 + 1 + 1 + 4 + 1 + 1 + 25,000 + 1 |
| Pull request | 10 | 2 + 3 + 4 + 1 |
| Issue | 6 | 1 + 1 + 4 |
| Issue comment | 8 | 2 + 1 + 2 + 1 + 1 + 1 |
| Review | 7 | 1 + 3 + 1 + 1 + 1 |
| Inline review comment | 6 | 1 + 4 + 1 |
| **All** | **25,063** | |

- 5 events have no actor: the two comments of repository 120, the two events of repository 300 and the inline review comment of repository 600. So 25,058 are attributable.
- People per row: 3 + 4 + 2 + 1 + 1 + 0 + 4 + 3 + 2 + 2 + 1 = 23.
- Top actor events, added up over the rows: 6 + 5 + 5 + 4 + 3 + 4 + 1 + 1 + 25,000 + 1 = 25,030. The NULL of repository 300 is not added.
- Top 3 events, added up: 10 + 9 + 8 + 4 + 3 + 12 + 3 + 2 + 25,001 + 1 = 25,053.
- 1 row without any attributable event (repository 300). 3 rows with exactly one person (200 on both days, and 710).

In [0]:
actual_schema = spark.table(TEST_GOLD).schema.simpleString()
staging_schema = spark.table(TEST_STAGING).schema.simpleString()

expected_schema = (
    "struct<repo_id:bigint,activity_date:date,repo_name:string,"
    "engineering_activity_events:bigint,attributable_events:bigint,unattributable_events:bigint,"
    "actor_coverage_pct:double,unique_active_actors:bigint,"
    "top_actor_events:bigint,top_actor_share_pct:double,"
    "top_3_actor_events:bigint,top_3_actor_share_pct:double,"
    "hhi_actor_activity:double,"
    "_gold_run_id:string,_gold_updated_at:timestamp>"
)

print("Gold    :", actual_schema)
print("Staging :", staging_schema)
print("Expected:", expected_schema)

# The three texts are long. The check table below gets a short text for each comparison.
if actual_schema == expected_schema:
    gold_schema_result = "as in the contract"
else:
    gold_schema_result = "DIFFERENT, see the lines above"

if staging_schema == expected_schema:
    staging_schema_result = "as in the contract"
else:
    staging_schema_result = "DIFFERENT, see the lines above"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 columns and types of the Gold table' AS check_name,
               '{gold_schema_result}' AS actual, 'as in the contract' AS expected
        UNION ALL
        SELECT '02 columns and types of the staging table', '{staging_schema_result}', 'as in the contract'
        UNION ALL
        SELECT '03 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 different days', CAST(COUNT(DISTINCT activity_date) AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows written by this run', CAST(count_if(_gold_run_id = 'test-run1' AND _gold_updated_at IS NOT NULL) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 sum of engineering_activity_events', CAST(SUM(engineering_activity_events) AS STRING), '25063' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 rows of repository 400 (only watch, fork and release)', CAST(count_if(repo_id = 400) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 repository 100 on 1 February (6, 3 and 1 events): counts', MAX(count_summary), 'all=10 attributable=10 unattributable=0 coverage=100.0 actors=3'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '09 repository 100 on 1 February (6, 3 and 1 events): top', MAX(top_summary), 'top=6 top_share=60.0 top_3=10 top_3_share=100.0 hhi=0.46 name=org/alpha-renamed'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '10 repository 110 on 1 February (5, 3, 1 and 1 events): counts', MAX(count_summary), 'all=10 attributable=10 unattributable=0 coverage=100.0 actors=4'
        FROM gold_summary WHERE repo_id = 110 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 repository 110 on 1 February (5, 3, 1 and 1 events): top', MAX(top_summary), 'top=5 top_share=50.0 top_3=9 top_3_share=90.0 hhi=0.36 name=org/beta'
        FROM gold_summary WHERE repo_id = 110 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '12 repository 120 on 1 February (2 without actor): counts', MAX(count_summary), 'all=10 attributable=8 unattributable=2 coverage=80.0 actors=2'
        FROM gold_summary WHERE repo_id = 120 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '13 repository 120 on 1 February (2 without actor): top', MAX(top_summary), 'top=5 top_share=62.5 top_3=8 top_3_share=100.0 hhi=0.5313 name=org/gamma'
        FROM gold_summary WHERE repo_id = 120 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '14 repository 200 on 1 February (one person): counts', MAX(count_summary), 'all=4 attributable=4 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '15 repository 200 on 1 February (one person): top', MAX(top_summary), 'top=4 top_share=100.0 top_3=4 top_3_share=100.0 hhi=1.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '16 repository 200 on 2 February (one person, three kinds): counts', MAX(count_summary), 'all=3 attributable=3 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '17 repository 200 on 2 February (one person, three kinds): top', MAX(top_summary), 'top=3 top_share=100.0 top_3=3 top_3_share=100.0 hhi=1.0 name=org/delta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '18 repository 300 on 2 February (nothing attributable): counts', MAX(count_summary), 'all=2 attributable=0 unattributable=2 coverage=0.0 actors=0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '19 repository 300 on 2 February (nothing attributable): top', MAX(top_summary), 'top=NULL top_share=NULL top_3=NULL top_3_share=NULL hhi=NULL name=NULL'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '20 repository 500 on 5 February (four-way tie): counts', MAX(count_summary), 'all=16 attributable=16 unattributable=0 coverage=100.0 actors=4'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '21 repository 500 on 5 February (four-way tie): top', MAX(top_summary), 'top=4 top_share=25.0 top_3=12 top_3_share=75.0 hhi=0.25 name=org/epsilon'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '22 repository 510 on 5 February (1 event each): counts', MAX(count_summary), 'all=3 attributable=3 unattributable=0 coverage=100.0 actors=3'
        FROM gold_summary WHERE repo_id = 510 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '23 repository 510 on 5 February (1 event each): top', MAX(top_summary), 'top=1 top_share=33.33 top_3=3 top_3_share=100.0 hhi=0.3333 name=org/zeta'
        FROM gold_summary WHERE repo_id = 510 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '24 repository 600 on 5 February (label, coverage 66.67): counts', MAX(count_summary), 'all=3 attributable=2 unattributable=1 coverage=66.67 actors=2'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '25 repository 600 on 5 February (label, coverage 66.67): top', MAX(top_summary), 'top=1 top_share=50.0 top_3=2 top_3_share=100.0 hhi=0.5 name=org/eta-new'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '26 repository 700 on 5 February (25,000 pushes): counts', MAX(count_summary), 'all=25001 attributable=25001 unattributable=0 coverage=100.0 actors=2'
        FROM gold_summary WHERE repo_id = 700 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '27 repository 700 on 5 February (25,000 pushes): top', MAX(top_summary), 'top=25000 top_share=100.0 top_3=25001 top_3_share=100.0 hhi=0.9999 name=org/busy'
        FROM gold_summary WHERE repo_id = 700 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '28 repository 710 on 5 February (a single push): counts', MAX(count_summary), 'all=1 attributable=1 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 710 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '29 repository 710 on 5 February (a single push): top', MAX(top_summary), 'top=1 top_share=100.0 top_3=1 top_3_share=100.0 hhi=1.0 name=org/eta-single'
        FROM gold_summary WHERE repo_id = 710 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '30 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '31 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '32 audit source_date', CAST(MAX(source_date) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '33 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '34 audit Silver rows read', MAX(source_summary), 'push=25026 pull_requests=10 issues=6 issue_comments=8 reviews=7 review_comments=6'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '35 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '6'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '36 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '37 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '38 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '39 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '40 audit has a table version', CAST(MAX(gold_table_version) IS NOT NULL AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '41 audit pipeline_name', MAX(pipeline_name), 'gold_repository_contributor_concentration'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '42 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '43 audit run times are in order', CAST(MAX(run_finished_at >= run_started_at) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '44 audit quality_metrics', MAX(metrics_summary), 'repository_days=11 all=25063 attributable=25058 unattributable=5 actors=23 top=25030 top_3=25053 without_attributable=1 single_actor=3'
        FROM audit_summary WHERE run_id = 'test-run1'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T7"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T7: {failed_count} check(s) failed. See the table above.")
print("T7: ALL CHECKS PASSED")

## T8. Run 2: the same run again

First the Gold rows of run 1 are saved as a copy, so that run 2 can be compared with it. The copy holds the key and the two summary lines of every row.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_COPY} AS
    SELECT repo_id, activity_date, count_summary, top_summary
    FROM gold_summary
""")

print("Copy of the Gold rows after run 1:", TEST_COPY)

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="" $run_id="test-run2"

## T9. Checks of run 2: nothing changed

`EXCEPT` returns the rows of the first query that are not in the second one. Both directions must return no rows. The two technical columns are not part of the summary, because they are new in every run.

The ranking gives ties the same order on every run, so the four-way tie of repository 500 must not change anything either.

A run for all dates replaces the whole table, so every row now carries the ID of run 2.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '02 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '03 rows of run 2 that are not in run 1', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, count_summary, top_summary FROM gold_summary
            EXCEPT
            SELECT repo_id, activity_date, count_summary, top_summary FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '04 rows of run 1 that are not in run 2', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, count_summary, top_summary FROM {TEST_COPY}
            EXCEPT
            SELECT repo_id, activity_date, count_summary, top_summary FROM gold_summary
        )
        UNION ALL
        SELECT '05 rows written by run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(metrics_summary), 'repository_days=11 all=25063 attributable=25058 unattributable=5 actors=23 top=25030 top_3=25053 without_attributable=1 single_actor=3'
        FROM audit_summary WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '08 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '3' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T9"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T9: {failed_count} check(s) failed. See the table above.")
print("T9: ALL CHECKS PASSED")

## T10. Run 3: late events, for one source date

Five new Silver rows arrive with the source date 2 February, in five different tables:

| Event | Table | Repository | Event date | What it tests |
|---|---|---|---|---|
| ...80 | Issue comment | 200 | 2 Feb | A second person (8) on a day that already has a Gold row |
| ...85 | Review | 200 | 2 Feb | A third person (9) on that day. Person 7 has more events on 1 February than anybody on 2 February, so the people must be ranked for each day on its own |
| ...81 | Push | 800 | 2 Feb | A repository that had no Gold row yet |
| ...82 | Pull request | 200 | **1 Feb** | A late event: it was loaded with the source date 2 February, but it happened on 1 February at 23:59. Only the pull request table says that 1 February has to be rebuilt. Person 7 already pushed that day |
| ...83 | Watch | 400 | **3 Feb** | A late watch event. It is not engineering work, so it must **not** make 3 February an affected day |

The run gets `source_date = 2024-02-02`.

In [0]:
spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000080', TIMESTAMP'2024-02-02 11:00:00', DATE'2024-02-02', 8,  200, 'org/delta-renamed', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_REVIEW + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000085', TIMESTAMP'2024-02-02 11:30:00', DATE'2024-02-02', 9,  200, 'org/delta-renamed', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000081', TIMESTAMP'2024-02-02 12:00:00', DATE'2024-02-02', 60, 800, 'org/theta',         DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_PULL_REQUEST + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000082', TIMESTAMP'2024-02-01 23:59:00', DATE'2024-02-01', 7,  200, 'org/delta',         DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
VALUES
    ('33000000083', TIMESTAMP'2024-02-03 00:30:00', DATE'2024-02-03', 16, 400, 'org/stars',         DATE'2024-02-02')
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="2024-02-02" $run_id="test-run3"

## T11. Checks of run 3

Worked out by hand.

- In the six tables, the source date 2 February has 9 rows now: the five of T5.2 (...46 to ...50) and the four new ones (...80, ...81, ...82 and ...85). Their event dates are 1 and 2 February, so **two** days are affected. The late watch event ...83 is not in the six tables, so 3 February is not affected. The six rows without a date belong to the source date 1 February and are not in the scope of this run.
- Both days are rebuilt from **all** their rows: 35 rows on 1 February (34 + the late one) and 8 rows on 2 February. 43 rows, 7 repository-days.
- By table: 20 pushes, 6 pull requests, 2 issues, 8 issue comments, 6 reviews and 1 inline review comment.
- Repository 200 on 1 February has 5 events now, all by person 7: still 1 person, top 5, HHI 1. The late event is the latest of that day, and its name is `org/delta`.
- Repository 200 on 2 February has 5 events: 3 by person 7 and 1 each by persons 8 and 9. 3 people. Top 3 of 5, share 60. Top 3: 3 + 1 + 1 = 5, share 100. HHI: (9 + 1 + 1) ÷ 25 = 0.44.
- The people are ranked for each day on its own. Person 7 has 5 events on 1 February, but that day does not push anybody of 2 February out of the top 3.
- Repository 800 on 2 February is new: one push by one person.
- The other rows of 1 and 2 February have the same numbers as before, but they were written again by this run.
- The five rows of 5 February are not touched. They still carry the ID of run 2.
- Gold has 7 + 5 = 12 rows.
- The quality numbers: 39 attributable events, 4 unattributable. People: 3 + 4 + 2 + 1 + 3 + 0 + 1 = 14. Top: 6 + 5 + 5 + 5 + 3 + 1 = 25. Top 3: 10 + 9 + 8 + 5 + 5 + 1 = 38. 1 row without attributable events, 2 rows with one person (200 on 1 February and 800).

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-02'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '04 audit affected_dates (no 3 February)', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary), 'push=20 pull_requests=6 issues=2 issue_comments=8 reviews=6 review_comments=1'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '07 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '7'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '08 Gold rows', CAST(COUNT(*) AS STRING), '12' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows of 1 and 2 February written by run 3', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run3') AS STRING), '7' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 rows of 3 February', CAST(count_if(activity_date = DATE'2024-02-03') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 repository 200 on 1 February (late event): counts', MAX(count_summary), 'all=5 attributable=5 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '13 repository 200 on 1 February (late event): top', MAX(top_summary), 'top=5 top_share=100.0 top_3=5 top_3_share=100.0 hhi=1.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '14 repository 200 on 2 February (three people): counts', MAX(count_summary), 'all=5 attributable=5 unattributable=0 coverage=100.0 actors=3'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '15 repository 200 on 2 February (three people): top', MAX(top_summary), 'top=3 top_share=60.0 top_3=5 top_3_share=100.0 hhi=0.44 name=org/delta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '16 repository 800 on 2 February (new row): counts', MAX(count_summary), 'all=1 attributable=1 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 800 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '17 repository 800 on 2 February (new row): top', MAX(top_summary), 'top=1 top_share=100.0 top_3=1 top_3_share=100.0 hhi=1.0 name=org/theta'
        FROM gold_summary WHERE repo_id = 800 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '18 repository 100 on 1 February (same numbers): counts', MAX(count_summary), 'all=10 attributable=10 unattributable=0 coverage=100.0 actors=3'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '19 repository 100 on 1 February (same numbers): top', MAX(top_summary), 'top=6 top_share=60.0 top_3=10 top_3_share=100.0 hhi=0.46 name=org/alpha-renamed'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '20 repository 300 on 2 February (same numbers): counts', MAX(count_summary), 'all=2 attributable=0 unattributable=2 coverage=0.0 actors=0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '21 repository 300 on 2 February (same numbers): top', MAX(top_summary), 'top=NULL top_share=NULL top_3=NULL top_3_share=NULL hhi=NULL name=NULL'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '22 repository 500 on 5 February (not touched): top', MAX(top_summary), 'top=4 top_share=25.0 top_3=12 top_3_share=75.0 hhi=0.25 name=org/epsilon'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '23 audit quality_metrics', MAX(metrics_summary), 'repository_days=7 all=43 attributable=39 unattributable=4 actors=14 top=25 top_3=38 without_attributable=1 single_actor=2'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '24 different keys in Gold', CAST(COUNT(DISTINCT repo_id, activity_date) AS STRING), '12' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T11"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T11: {failed_count} check(s) failed. See the table above.")
print("T11: ALL CHECKS PASSED")

## T12. Run 4: removed events

Three Silver rows are removed, from three different tables: the comment of person 8 and the review of person 9 in repository 200 on 2 February (...80, ...85), and the only event of repository 800 (...81). Then the source date 2 February runs again.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_ISSUE_COMMENT}
    WHERE event_id = '33000000080'
""")

spark.sql(f"""
    DELETE FROM {TEST_REVIEW}
    WHERE event_id = '33000000085'
""")

spark.sql(f"""
    DELETE FROM {TEST_PUSH}
    WHERE event_id = '33000000081'
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="2024-02-02" $run_id="test-run4"

## T13. Checks of run 4

Worked out by hand.

- In the six tables, the source date 2 February has 6 rows left: ...46 to ...50 and the late event ...82. The affected days are still 1 and 2 February.
- 35 rows on 1 February and 5 rows on 2 February: 40 rows, 6 repository-days.
- By table: 19 pushes, 6 pull requests, 2 issues, 7 issue comments, 5 reviews and 1 inline review comment.
- Repository 200 on 2 February is back to 3 events by one person: HHI 1 again.
- Repository 800 has no event any more, so its Gold row is gone.
- Gold has 6 + 5 = 11 rows. The rows of 5 February still carry the ID of run 2.
- The quality numbers: 36 attributable, 4 unattributable, 3 + 4 + 2 + 1 + 1 + 0 = 11 people, top 6 + 5 + 5 + 5 + 3 = 24, top 3 10 + 9 + 8 + 5 + 3 = 35.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '02 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '03 audit Silver rows read', MAX(source_summary), 'push=19 pull_requests=6 issues=2 issue_comments=7 reviews=5 review_comments=1'
        FROM audit_summary WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '04 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '6'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of repository 800 (removed)', CAST(count_if(repo_id = 800) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 200 on 2 February (lowered): counts', MAX(count_summary), 'all=3 attributable=3 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '08 repository 200 on 2 February (lowered): top', MAX(top_summary), 'top=3 top_share=100.0 top_3=3 top_3_share=100.0 hhi=1.0 name=org/delta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '09 repository 200 on 1 February (late event still there): counts', MAX(count_summary), 'all=5 attributable=5 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '10 repository 200 on 1 February (late event still there): top', MAX(top_summary), 'top=5 top_share=100.0 top_3=5 top_3_share=100.0 hhi=1.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 rows of 1 and 2 February written by run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '6' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 audit quality_metrics', MAX(metrics_summary), 'repository_days=6 all=40 attributable=36 unattributable=4 actors=11 top=24 top_3=35 without_attributable=1 single_actor=2'
        FROM audit_summary WHERE run_id = 'test-run4'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T13"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T13: {failed_count} check(s) failed. See the table above.")
print("T13: ALL CHECKS PASSED")

## T14. Run 5: a source date without rows in the six tables

No row of the six tables has the source date 3 February. The watch table has one (...54), but this notebook does not read the watch table. The run gets `source_date = 2024-02-03`.

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="2024-02-03" $run_id="test-run5"

## T15. Checks of run 5: `NO_INPUT`, Gold unchanged

There is no affected day, so nothing is published. The Gold table still holds the 6 rows of run 4 and the 5 rows of run 2. The run still writes its audit row.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'NO_INPUT' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-03'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '04 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary), 'push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 review_comments=0'
        FROM audit_summary WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '07 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '08 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '09 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '10 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows written by run 5', CAST(count_if(_gold_run_id = 'test-run5') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 Gold rows still from run 4', CAST(count_if(_gold_run_id = 'test-run4') AS STRING), '6' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 Gold rows still from run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '5' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T15"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T15: {failed_count} check(s) failed. See the table above.")
print("T15: ALL CHECKS PASSED")

## T16. Run 6: quality gates fail

The Silver notebooks never let such rows into Silver. The test puts two of them into the test tables anyway, both on 5 February:

| Bad row | Table | What is wrong | The gate that must catch it |
|---|---|---|---|
| ...84 | Push | No `repo_id`. Its group has no repository, so the Gold key would be NULL | `KEY_IS_NULL` |
| ...74 again | Issue | An issue in repository 710 with the `event_id` of the pull request ...74. One event ID is now in two tables | `EVENT_ID_NOT_UNIQUE` |

The row without a repository also makes a third gate fail, `CONCENTRATION_DOES_NOT_FIT_ITS_BASE`. Step 5.5 joins the top counts on `repo_id`, and in SQL `NULL = NULL` is not true. So the row finds no top counts, although it has an attributable event. That is right: the row is broken, and a gate says so.

The other gates check the arithmetic of the notebook itself, for example that attributable and unattributable events add up to all events. No Silver row can make them fail.

The cell below prepares the faults. The run itself is in the cell after it.

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
VALUES
    -- A row without repo_id
    ('33000000084', TIMESTAMP'2024-02-05 13:00:00', DATE'2024-02-05', 70, NULL, 'org/broken',     DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_ISSUE + " " + SEVEN_COLUMNS + """
VALUES
    -- The event ID of the pull request ...74
    ('33000000074', TIMESTAMP'2024-02-05 14:00:00', DATE'2024-02-05', 71, 710,  'org/eta-single', DATE'2024-02-05')
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

### Run 6

**The next cell is EXPECTED to fail** with `RuntimeError: repository_contributor_concentration FAILED: Quality gates failed: KEY_IS_NULL; CONCENTRATION_DOES_NOT_FIT_ITS_BASE; EVENT_ID_NOT_UNIQUE`. After it has failed, click the cell after it and choose "Run all below".

If this cell runs twice, the audit table gets two rows for run 6. The checks count runs, not rows, so that does no harm.

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="2024-02-05" $run_id="test-run6-fail"

## T17. Checks of run 6: Gold is unchanged

Worked out by hand.

- In the six tables, the source date 5 February has 25,026 rows: the 25,024 good ones and the 2 bad ones. The affected day is 5 February.
- By table: 25,008 pushes, 5 pull requests, 5 issues, 1 issue comment, 2 reviews and 5 inline review comments.
- The run staged 6 rows: repositories 500, 510, 600, 700 and 710, and one row without a repository.
- The staged row of repository 710 has 2 events by 2 people: its push and the issue with the repeated ID. Top 1 of 2, share 50, HHI (1 + 1) ÷ 4 = 0.5.
- The staged row without a repository has 1 attributable event by 1 person, but no top count and no label.
- The three gates are named in the order in which the notebook checks them.
- The run must have written its audit row with the status `FAILED` and the names of the gates.
- The Gold table must be exactly what it was: 11 rows, and the rows of 5 February still carry the ID of run 2. The rejected result is only in the staging table.

After the checks, the test removes the two bad rows, so that the next run can succeed.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'FAILED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), 'KEY_IS_NULL; CONCENTRATION_DOES_NOT_FIT_ITS_BASE; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '03 audit error_message', MAX(error_message), 'Quality gates failed: KEY_IS_NULL; CONCENTRATION_DOES_NOT_FIT_ITS_BASE; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '05 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '06 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '07 audit Silver rows read', MAX(source_summary), 'push=25008 pull_requests=5 issues=5 issue_comments=1 reviews=2 review_comments=5'
        FROM audit_summary WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '08 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 Gold rows written by the failed run', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 Gold rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 staging rows of the rejected result', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '6' FROM {TEST_STAGING}
        UNION ALL
        SELECT '13 staging rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '1' FROM {TEST_STAGING}
        UNION ALL
        SELECT '14 staging row of repository 710: its push and the repeated issue',
               MAX(concat('all=', CAST(engineering_activity_events AS STRING),
                          ' actors=', CAST(unique_active_actors AS STRING),
                          ' top=', CAST(top_actor_events AS STRING),
                          ' top_share=', CAST(top_actor_share_pct AS STRING),
                          ' hhi=', CAST(hhi_actor_activity AS STRING))),
               'all=2 actors=2 top=1 top_share=50.0 hhi=0.5'
        FROM {TEST_STAGING} WHERE repo_id = 710
        UNION ALL
        SELECT '15 staging row without repo_id: no top count and no label',
               MAX(concat('attributable=', CAST(attributable_events AS STRING),
                          ' actors=', CAST(unique_active_actors AS STRING),
                          ' top=', COALESCE(CAST(top_actor_events AS STRING), 'NULL'),
                          ' name=', COALESCE(repo_name, 'NULL'))),
               'attributable=1 actors=1 top=NULL name=NULL'
        FROM {TEST_STAGING} WHERE repo_id IS NULL
        UNION ALL
        SELECT '16 Gold row of repository 710 (unchanged): counts', MAX(count_summary), 'all=1 attributable=1 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 710 AND activity_date = DATE'2024-02-05'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T17"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T17: {failed_count} check(s) failed. See the table above.")
print("T17: ALL CHECKS PASSED")

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_PUSH}
    WHERE event_id = '33000000084'
""")

# Only the issue with the repeated ID is removed. The pull request ...74 stays.
spark.sql(f"""
    DELETE FROM {TEST_ISSUE}
    WHERE event_id = '33000000074'
""")

print("Rows in the Silver test tables after the repair:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

## T18. Run 7: recovery

The same run again, now without the bad rows. The five rows of 5 February are replaced. The six rows of 1 and 2 February are not touched and still carry the ID of run 4.

The quality numbers of the five rows: 25,024 events, 25,023 of them attributable. People: 4 + 3 + 2 + 2 + 1 = 12. Top: 4 + 1 + 1 + 25,000 + 1 = 25,007. Top 3: 12 + 3 + 2 + 25,001 + 1 = 25,019. 1 row with one person (710).

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="2024-02-05" $run_id="test-run7"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '02 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '5'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '03 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '04 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of 5 February written by run 7', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run7') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 rows of 1 and 2 February still from run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '6' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 repository 500 on 5 February: top', MAX(top_summary), 'top=4 top_share=25.0 top_3=12 top_3_share=75.0 hhi=0.25 name=org/epsilon'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '09 repository 600 on 5 February: top', MAX(top_summary), 'top=1 top_share=50.0 top_3=2 top_3_share=100.0 hhi=0.5 name=org/eta-new'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '10 repository 700 on 5 February: counts', MAX(count_summary), 'all=25001 attributable=25001 unattributable=0 coverage=100.0 actors=2'
        FROM gold_summary WHERE repo_id = 700 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '11 repository 710 on 5 February: counts', MAX(count_summary), 'all=1 attributable=1 unattributable=0 coverage=100.0 actors=1'
        FROM gold_summary WHERE repo_id = 710 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '12 repository 710 on 5 February: top', MAX(top_summary), 'top=1 top_share=100.0 top_3=1 top_3_share=100.0 hhi=1.0 name=org/eta-single'
        FROM gold_summary WHERE repo_id = 710 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '13 audit Silver rows read', MAX(source_summary), 'push=25007 pull_requests=5 issues=4 issue_comments=1 reviews=2 review_comments=5'
        FROM audit_summary WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '14 audit quality_metrics', MAX(metrics_summary), 'repository_days=5 all=25024 attributable=25023 unattributable=1 actors=12 top=25007 top_3=25019 without_attributable=0 single_actor=1'
        FROM audit_summary WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '15 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '8' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T18"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Run 8: a day that no longer exists in Silver

All Silver rows of 5 February are removed from the nine tables. A run for the source date 5 February would now find no rows and end as `NO_INPUT`, as in T15, and the five Gold rows of that day would stay. The way to remove such a day is a run for all dates: it replaces the whole table.

In [0]:
for table_name in SILVER_TABLE_NAMES:
    spark.sql(f"""
        DELETE FROM {TEST_SILVER}.{table_name}
        WHERE event_date = DATE'2024-02-05'
    """)

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="" $run_id="test-run8"

## T20. Checks of run 8

Worked out by hand: the six tables have 46 rows left. 40 have an event date, on 1 and 2 February: 19 pushes, 6 pull requests, 2 issues, 7 issue comments, 5 reviews and 1 inline review comment. 6 have none. Gold has the 6 rows of those two days, with the numbers of run 4, and no row of 5 February.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '03 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '04 audit Silver rows read', MAX(source_summary), 'push=19 pull_requests=6 issues=2 issue_comments=7 reviews=5 review_comments=1'
        FROM audit_summary WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '05 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '6'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '6'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '07 Gold rows', CAST(COUNT(*) AS STRING), '6' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 Gold rows of 5 February', CAST(count_if(activity_date = DATE'2024-02-05') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows written by run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '6' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 audit quality_metrics', MAX(metrics_summary), 'repository_days=6 all=40 attributable=36 unattributable=4 actors=11 top=24 top_3=35 without_attributable=1 single_actor=2'
        FROM audit_summary WHERE run_id = 'test-run8'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T20"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T20: {failed_count} check(s) failed. See the table above.")
print("T20: ALL CHECKS PASSED")

## T21. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="2024-2-3" $run_id="test-bad-date"

### T21 check: where did it stop?

`%run` runs the other notebook inside this one, so the variables of that notebook can be read here. They show how far the stopped run came:

- `source_date` holds the bad value, so step 1.2 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4, where it reads the run ID. It stopped before it touched a table.

In [0]:
print("source_date:", source_date)
print("run_id     :", run_id)

if source_date != "2024-2-3":
    raise AssertionError("T21: the run with the bad source_date has not run. Run the cell above first.")

if run_id == "test-bad-date":
    raise AssertionError("T21: the notebook did not stop at the bad source_date. It went on to step 1.4.")

section_results["T21"] = 0
print("T21: the notebook stopped at the bad source_date, as expected")

## T22. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../03_repository_contributor_concentration $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema"

### T22 check: where did it stop?

- `test_schema` holds the bad value, so step 1.3 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if test_schema != "TEST_WRONG":
    raise AssertionError("T22: the run with the bad test_schema has not run. Run the cell above first.")

if run_id == "test-bad-schema":
    raise AssertionError("T22: the notebook did not stop at the bad test_schema. It went on to step 1.4.")

section_results["T22"] = 0
print("T22: the notebook stopped at the bad test_schema, as expected")

## T23. Bad parameter: `run_id`

The run ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`. `bad.run.id` contains dots.

**The next cell is EXPECTED to fail** with `ValueError: Invalid run_id 'bad.run.id'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_g3" $source_date="" $run_id="bad.run.id"

### T23 check: did it run?

- `run_id` holds the bad value, so the notebook came to step 1.4.
- That it stopped there, and wrote nothing, is checked in T24: this run has no audit row and the Gold table is still the one of run 8.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if run_id != "bad.run.id":
    raise AssertionError("T23: the run with the bad run_id has not run. Run the cell above first.")

section_results["T23"] = 0
print("T23: the notebook came to the run_id step. T24 checks that it wrote nothing.")

## T24. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so the audit table still holds 9 runs and the Gold table is still the one of run 8.
- Every run wrote exactly one audit row. Run 6 is left out of that check, because its cell may have run more than once.
- The versions of the production tables are read again and compared with the versions from T2.

In [0]:
prod_audit_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_after = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_after = -1

print("Production audit version  :", prod_audit_version_after)
print("Production Gold version   :", prod_gold_version_after)
print("Production staging version:", prod_staging_version_after)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 different runs in the audit table' AS check_name, CAST(COUNT(DISTINCT run_id) AS STRING) AS actual, '9' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 audit rows of the three stopped runs', CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'bad.run.id')) AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '03 Gold rows still from run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '6'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '6'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 runs by status',
               concat('SUCCEEDED=', CAST(COUNT(DISTINCT CASE WHEN status = 'SUCCEEDED' THEN run_id END) AS STRING),
                      ' FAILED=', CAST(COUNT(DISTINCT CASE WHEN status = 'FAILED' THEN run_id END) AS STRING),
                      ' NO_INPUT=', CAST(COUNT(DISTINCT CASE WHEN status = 'NO_INPUT' THEN run_id END) AS STRING)),
               'SUCCEEDED=6 FAILED=1 NO_INPUT=2'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '06 production audit table version unchanged', '{prod_audit_version_after}', '{prod_audit_version_before}'
        UNION ALL
        SELECT '07 production Gold table version unchanged', '{prod_gold_version_after}', '{prod_gold_version_before}'
        UNION ALL
        SELECT '08 production staging version unchanged', '{prod_staging_version_after}', '{prod_staging_version_before}'
        UNION ALL
        SELECT '09 audit rows of the 8 runs without run 6 (one row each)', CAST(count_if(run_id <> 'test-run6-fail') AS STRING), '8'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '10 audit rows of run 6 are all FAILED', CAST(count_if(run_id = 'test-run6-fail' AND status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '11 audit rows with another pipeline or table name', CAST(count_if(pipeline_name <> 'gold_repository_contributor_concentration' OR gold_table <> '{TEST_GOLD}') AS STRING), '0'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T24"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T24: {failed_count} check(s) failed. See the table above.")
print("T24: ALL CHECKS PASSED")

## Result of the whole test

Every section wrote its number of failed checks into `section_results`. This cell goes through the 13 sections. A section that is not in the dictionary did not run, for example because the cells above it were skipped.

The test has passed only when this cell prints `TEST PASSED`.

In [0]:
EXPECTED_SECTIONS = [
    "T4", "T7", "T9", "T11", "T13", "T15", "T17",
    "T18", "T20", "T21", "T22", "T23", "T24",
]

problems = []

for section in EXPECTED_SECTIONS:
    if section not in section_results:
        problems.append(section + " did not run")
    elif section_results[section] > 0:
        problems.append(section + " has " + str(section_results[section]) + " failed check(s)")

print("Sections that ran:", len(section_results), "of", len(EXPECTED_SECTIONS))

if len(problems) > 0:
    raise AssertionError("TEST FAILED: " + "; ".join(problems))

print("TEST PASSED: all", len(EXPECTED_SECTIONS), "sections ran and every check passed.")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

If the test failed, you can leave this cell out and look at the test tables first. The next test run drops the schema anyway, in T1.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)